# Evaluate RobustMultimodalClassifier on June 17 new data

This notebook loads the provided robust multimodal CNN architecture, applies the saved scalers from the trained model folder, predicts the parsed Bacillus/Micrococcus June 17 data, and exports metrics, predictions, confusion matrices, and composition summaries.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import warnings

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis"
)

MODEL_DIR = PROJECT_ROOT / "models" / "trained" / "exp06_robust_multimodal_species_v1"

PARSED_DIR = PROJECT_ROOT / "data" / "live_rapid_e" / "parsed"

RESULTS_DIR = PROJECT_ROOT / "results" / "live_rapid_e" / "robust_multimodal_new_data_evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "model.pt"

COMBINED_PARQUET = PARSED_DIR / "june17_bacillus_cereus_possible_micrococcus_preprocessed.parquet"

INDIVIDUAL_PARQUETS = [
    PARSED_DIR / "june17_bacillus_cereus_possible_preprocessed.parquet",
    PARSED_DIR / "june17_micrococcus_preprocessed.parquet",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Model dir:", MODEL_DIR)
print("Model path:", MODEL_PATH)
print("Parsed dir:", PARSED_DIR)
print("Results dir:", RESULTS_DIR)
print("Device:", DEVICE)


Project root: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis
Model dir: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained\exp06_robust_multimodal_species_v1
Model path: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained\exp06_robust_multimodal_species_v1\model.pt
Parsed dir: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\data\live_rapid_e\parsed
Results dir: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\robust_multimodal_new_data_evaluation
Device: cpu


## 1. Define the model architecture

In [2]:
class ConvBranch1D(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        out_dim: int = 64,
        dropout: float = 0.25,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),

            nn.Dropout(dropout),
            nn.Linear(128, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class ScalarBranch(nn.Module):
    def __init__(
        self,
        input_dim: int = 2,
        out_dim: int = 16,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.1),

            nn.Linear(16, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class RobustMultimodalClassifier(nn.Module):
    def __init__(
        self,
        n_classes: int,
        branch_dim: int = 64,
        scalar_dim: int = 16,
        modality_dropout_p: float = 0.15,
    ):
        super().__init__()

        self.modality_dropout_p = modality_dropout_p

        self.spectrometer_branch = ConvBranch1D(
            out_dim=branch_dim,
            dropout=0.25,
        )

        self.lifetime_branch = ConvBranch1D(
            out_dim=branch_dim,
            dropout=0.25,
        )

        self.scattering_branch = ConvBranch1D(
            out_dim=branch_dim,
            dropout=0.25,
        )

        self.scalar_branch = ScalarBranch(
            input_dim=2,
            out_dim=scalar_dim,
        )

        fusion_dim = branch_dim * 3 + scalar_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.35),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(64, n_classes),
        )

    def _maybe_drop(self, z):
        if (
            not self.training
            or self.modality_dropout_p <= 0
        ):
            return z

        if torch.rand(1, device=z.device).item() < self.modality_dropout_p:
            return torch.zeros_like(z)

        return z

    def forward(
        self,
        batch,
        enabled_modalities: Iterable[str] | None = None,
    ):
        enabled = set(
            enabled_modalities
            or [
                "spectrometer",
                "lifetime",
                "scattering",
                "scalar",
            ]
        )

        z_spec = self.spectrometer_branch(batch["spectrometer"])
        z_life = self.lifetime_branch(batch["lifetime"])
        z_scat = self.scattering_branch(batch["scattering"])
        z_scalar = self.scalar_branch(batch["scalar"])

        if "spectrometer" not in enabled:
            z_spec = torch.zeros_like(z_spec)

        if "lifetime" not in enabled:
            z_life = torch.zeros_like(z_life)

        if "scattering" not in enabled:
            z_scat = torch.zeros_like(z_scat)

        if "scalar" not in enabled:
            z_scalar = torch.zeros_like(z_scalar)

        z_spec = self._maybe_drop(z_spec)
        z_life = self._maybe_drop(z_life)
        z_scat = self._maybe_drop(z_scat)
        z_scalar = self._maybe_drop(z_scalar)

        z = torch.cat(
            [
                z_spec,
                z_life,
                z_scat,
                z_scalar,
            ],
            dim=1,
        )

        return self.classifier(z)


## 2. Load parsed new data

In [3]:
def load_new_data() -> pd.DataFrame:
    if COMBINED_PARQUET.exists():
        print("Loading combined parquet:", COMBINED_PARQUET)
        df = pd.read_parquet(COMBINED_PARQUET)
    else:
        existing = [p for p in INDIVIDUAL_PARQUETS if p.exists()]
        if not existing:
            raise FileNotFoundError(
                "Could not find combined parquet or individual June 17 parsed parquet files."
            )
        print("Loading individual parquet files:")
        for p in existing:
            print(" ", p)
        df = pd.concat([pd.read_parquet(p) for p in existing], ignore_index=True)

    return df


df = load_new_data()
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()


Loading combined parquet: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\data\live_rapid_e\parsed\june17_bacillus_cereus_possible_micrococcus_preprocessed.parquet
Rows: 53
Columns: 1977


,raw_file,raw_path,particle_index,timestamp,serial,version,number_of_modules,has_fluorescence,lt11_framelength,lt03_framelength,...,si_1433,si_1434,si_1435,si_1436,si_1437,si_1438,si_1439,experiment_date,expected_sample,expected_composition
0,D_000000155_202606171224.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,289,2026-06-17 14:24:10.359,3532891,2,3,True,960,256,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible
1,D_000000156_202606171225.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,265,2026-06-17 14:25:13.295,3532891,2,3,True,936,256,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible
2,D_000000156_202606171225.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,1156,2026-06-17 14:25:44.207,3532891,2,3,True,13680,256,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible
3,D_000000156_202606171225.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,2387,2026-06-17 14:25:59.660,3532891,2,3,True,4296,256,...,0.001316,0.0,0.0,0.0,0.0,0.011194,0.0,2026-06-17,B_cereus,B_cereus_possible
4,D_000000158_202606171227.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,2115,2026-06-17 14:27:26.556,3532891,2,3,True,912,256,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible


In [4]:
def normalize_label(x) -> str:
    x = str(x)

    mapping = {
        "B. cereus": "B_cereus",
        "Bacillus cereus": "B_cereus",
        "bacillus_cereus_possible": "B_cereus",
        "B_cereus_possible": "B_cereus",
        "B_cereus": "B_cereus",

        "M. luteus": "M_luteus",
        "Micrococcus luteus": "M_luteus",
        "micrococcus": "M_luteus",
        "M_luteus": "M_luteus",
    }

    return mapping.get(x, x)


def add_ground_truth(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "expected_sample" in df.columns:
        source = df["expected_sample"]
    elif "experiment_name" in df.columns:
        source = df["experiment_name"]
    else:
        raise ValueError("Could not find expected_sample or experiment_name column for ground truth.")

    df["y_true"] = source.map(normalize_label)

    return df


df = add_ground_truth(df)

print(df["y_true"].value_counts(dropna=False))
cols = [c for c in ["experiment_name", "expected_sample", "y_true"] if c in df.columns]
df[cols].drop_duplicates()


y_true
B_cereus    28
M_luteus    25
Name: count, dtype: int64


,experiment_name,expected_sample,y_true
0,bacillus_cereus_possible,B_cereus,B_cereus
28,micrococcus,M_luteus,M_luteus


## 3. Build feature tensors

In [5]:
def sorted_feature_cols(df: pd.DataFrame, prefix: str) -> list[str]:
    cols = [c for c in df.columns if c.startswith(prefix)]

    def key_fn(c: str):
        suffix = c.replace(prefix, "")
        try:
            return int(suffix)
        except ValueError:
            return suffix

    return sorted(cols, key=key_fn)


FS_COLS = sorted_feature_cols(df, "fs_")
LT_COLS = sorted_feature_cols(df, "lt_")
SI_COLS = sorted_feature_cols(df, "si_")
SCALAR_COLS = [c for c in ["size", "time_asymmetry"] if c in df.columns]

print("fs columns:", len(FS_COLS))
print("lt columns:", len(LT_COLS))
print("si columns:", len(SI_COLS))
print("scalar columns:", SCALAR_COLS)

assert len(FS_COLS) == 256, f"Expected 256 fs_ columns, found {len(FS_COLS)}"
assert len(LT_COLS) == 256, f"Expected 256 lt_ columns, found {len(LT_COLS)}"
assert len(SI_COLS) == 1440, f"Expected 1440 si_ columns, found {len(SI_COLS)}"
assert len(SCALAR_COLS) == 2, f"Expected scalar columns ['size', 'time_asymmetry'], found {SCALAR_COLS}"


fs columns: 256
lt columns: 256
si columns: 1440
scalar columns: ['size', 'time_asymmetry']


In [6]:
def load_scaler(model_dir: Path, candidates: list[str]):
    for name in candidates:
        path = model_dir / name
        if path.exists():
            print("Loaded scaler:", path.name)
            return joblib.load(path)
    raise FileNotFoundError(f"Could not find scaler. Tried: {candidates}")


spectrometer_scaler = load_scaler(
    MODEL_DIR,
    ["spectrometer_scaler.joblib", "spectra_scaler.joblib"],
)
lifetime_scaler = load_scaler(
    MODEL_DIR,
    ["lifetime_scaler.joblib"],
)
scattering_scaler = load_scaler(
    MODEL_DIR,
    ["scattering_scaler.joblib"],
)
scalar_scaler = load_scaler(
    MODEL_DIR,
    ["scalar_scaler.joblib"],
)


Loaded scaler: spectrometer_scaler.joblib
Loaded scaler: lifetime_scaler.joblib
Loaded scaler: scattering_scaler.joblib
Loaded scaler: scalar_scaler.joblib


In [7]:
def build_inputs(df: pd.DataFrame) -> dict[str, torch.Tensor]:
    X_spec = df[FS_COLS].to_numpy(dtype=np.float32)
    X_life = df[LT_COLS].to_numpy(dtype=np.float32)
    X_scat = df[SI_COLS].to_numpy(dtype=np.float32)
    X_scalar = df[SCALAR_COLS].to_numpy(dtype=np.float32)

    X_spec = spectrometer_scaler.transform(X_spec).astype(np.float32)
    X_life = lifetime_scaler.transform(X_life).astype(np.float32)
    X_scat = scattering_scaler.transform(X_scat).astype(np.float32)
    X_scalar = scalar_scaler.transform(X_scalar).astype(np.float32)

    batch = {
        "spectrometer": torch.from_numpy(X_spec).unsqueeze(1),
        "lifetime": torch.from_numpy(X_life).unsqueeze(1),
        "scattering": torch.from_numpy(X_scat).unsqueeze(1),
        "scalar": torch.from_numpy(X_scalar),
    }

    return batch


inputs = build_inputs(df)

for name, tensor in inputs.items():
    print(name, tuple(tensor.shape), tensor.dtype)


spectrometer (53, 1, 256) torch.float32
lifetime (53, 1, 256) torch.float32
scattering (53, 1, 1440) torch.float32
scalar (53, 2) torch.float32


## 4. Load checkpoint and model

In [8]:
def load_checkpoint(model_path: Path) -> dict:
    if not model_path.exists():
        raise FileNotFoundError(model_path)

    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)

    if isinstance(ckpt, dict):
        return ckpt

    raise TypeError(f"Expected checkpoint dictionary, got {type(ckpt)}")


def class_names_from_checkpoint(ckpt: dict) -> list[str]:
    return ckpt.get(
        "class_names",
        ["B_cereus", "B_endophyticus", "K_salsicia", "M_luteus", "S_huminis"],
    )


def state_dict_from_checkpoint(ckpt: dict) -> dict:
    if "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]
    if "state_dict" in ckpt:
        return ckpt["state_dict"]

    tensor_values = [v for v in ckpt.values() if torch.is_tensor(v)]
    if tensor_values:
        return ckpt

    raise KeyError("Could not find model_state_dict or state_dict in checkpoint.")


ckpt = load_checkpoint(MODEL_PATH)
print("Checkpoint keys:", list(ckpt.keys()))

class_names = class_names_from_checkpoint(ckpt)
print("Class names:", class_names)

model = RobustMultimodalClassifier(n_classes=len(class_names))
state_dict = state_dict_from_checkpoint(ckpt)

model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)
model.eval()

print("Loaded model:", model.__class__.__name__)


Checkpoint keys: ['model_state_dict', 'n_classes', 'class_names', 'model_name', 'architecture', 'model_module', 'input_features', 'fluorescence_threshold', 'scattering_target_acquisitions', 'n_scattering_angles', 'rejection_config', 'distance_reference']
Class names: ['B. cereus', 'B. endophyticus', 'K. salsicia', 'M. luteus', 'S. huminis']
Loaded model: RobustMultimodalClassifier


## 5. Predict on new data

In [9]:
@torch.no_grad()
def predict_in_batches(
    model: nn.Module,
    inputs: dict[str, torch.Tensor],
    class_names: list[str],
    batch_size: int = 512,
    enabled_modalities: Iterable[str] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    n = next(iter(inputs.values())).shape[0]

    all_proba = []

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)

        batch = {
            key: value[start:end].to(DEVICE)
            for key, value in inputs.items()
        }

        logits = model(batch, enabled_modalities=enabled_modalities)
        proba = F.softmax(logits, dim=1).detach().cpu().numpy()
        all_proba.append(proba)

    proba = np.vstack(all_proba)
    pred_idx = proba.argmax(axis=1)
    pred_labels = np.array([class_names[i] for i in pred_idx])

    return pred_labels, proba


closed_pred, proba = predict_in_batches(
    model,
    inputs,
    class_names,
    batch_size=512,
)

print(pd.Series(closed_pred).value_counts())
print("proba shape:", proba.shape)


B. endophyticus    53
Name: count, dtype: int64
proba shape: (53, 5)


## 6. Optional robust rejection

In [10]:
def entropy_from_proba(proba: np.ndarray) -> np.ndarray:
    eps = 1e-12
    return -np.sum(proba * np.log(proba + eps), axis=1)


def apply_rejection(
    closed_pred: np.ndarray,
    proba: np.ndarray,
    ckpt: dict,
) -> np.ndarray:
    pred = closed_pred.copy()

    rejection_config = ckpt.get("rejection_config", {}) or {}

    softmax_threshold = rejection_config.get("softmax_threshold", None)
    margin_threshold = rejection_config.get("margin_threshold", None)
    entropy_threshold = rejection_config.get("entropy_threshold", None)

    max_prob = proba.max(axis=1)
    sorted_proba = np.sort(proba, axis=1)
    margin = sorted_proba[:, -1] - sorted_proba[:, -2]
    entropy = entropy_from_proba(proba)

    reject = np.zeros(len(pred), dtype=bool)

    if softmax_threshold is not None:
        reject |= max_prob < float(softmax_threshold)

    if margin_threshold is not None:
        reject |= margin < float(margin_threshold)

    if entropy_threshold is not None:
        reject |= entropy > float(entropy_threshold)

    pred[reject] = "unknown"

    print("Rejection config:", rejection_config)
    print("Rejected:", int(reject.sum()), "/", len(reject), f"({reject.mean():.2%})")

    return pred


robust_pred = apply_rejection(closed_pred, proba, ckpt)

print(pd.Series(robust_pred).value_counts())


Rejection config: {'softmax_threshold': 0.8, 'margin_threshold': 0.15, 'entropy_threshold': None, 'use_train_distance_rejection': True, 'train_distance_quantile': 0.995}
Rejected: 0 / 53 (0.00%)
B. endophyticus    53
Name: count, dtype: int64


## 7. Evaluate performance

In [11]:
def evaluate_predictions(y_true, y_pred, proba, model_id: str) -> dict:
    y_true = np.array([normalize_label(x) for x in y_true])
    y_pred = np.array([normalize_label(x) for x in y_pred])

    return {
        "model_id": model_id,
        "n_particles": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "unknown_fraction": float(np.mean(y_pred == "unknown")),
        "mean_confidence": float(proba.max(axis=1).mean()),
        "median_confidence": float(np.median(proba.max(axis=1))),
    }


metrics_df = pd.DataFrame([
    evaluate_predictions(df["y_true"], closed_pred, proba, "robust_multimodal_closed_set"),
    evaluate_predictions(df["y_true"], robust_pred, proba, "robust_multimodal_with_rejection"),
])

metrics_df


,model_id,n_particles,accuracy,balanced_accuracy,macro_f1,unknown_fraction,mean_confidence,median_confidence
0,robust_multimodal_closed_set,53,0.0,0.0,0.0,0.0,0.999572,1.0
1,robust_multimodal_with_rejection,53,0.0,0.0,0.0,0.0,0.999572,1.0


In [12]:
print("Closed-set classification report")
print(classification_report(
    df["y_true"].map(normalize_label),
    pd.Series(closed_pred).map(normalize_label),
    zero_division=0,
))

print("\nRobust/rejected classification report")
print(classification_report(
    df["y_true"].map(normalize_label),
    pd.Series(robust_pred).map(normalize_label),
    zero_division=0,
))


Closed-set classification report
                 precision    recall  f1-score   support

B. endophyticus       0.00      0.00      0.00       0.0
       B_cereus       0.00      0.00      0.00      28.0
       M_luteus       0.00      0.00      0.00      25.0

       accuracy                           0.00      53.0
      macro avg       0.00      0.00      0.00      53.0
   weighted avg       0.00      0.00      0.00      53.0


Robust/rejected classification report
                 precision    recall  f1-score   support

B. endophyticus       0.00      0.00      0.00       0.0
       B_cereus       0.00      0.00      0.00      28.0
       M_luteus       0.00      0.00      0.00      25.0

       accuracy                           0.00      53.0
      macro avg       0.00      0.00      0.00      53.0
   weighted avg       0.00      0.00      0.00      53.0



In [13]:
def make_confusion_df(y_true, y_pred) -> pd.DataFrame:
    labels = sorted(set(map(normalize_label, y_true)) | set(map(normalize_label, y_pred)))

    cm = confusion_matrix(
        [normalize_label(x) for x in y_true],
        [normalize_label(x) for x in y_pred],
        labels=labels,
    )

    return pd.DataFrame(cm, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])


closed_cm = make_confusion_df(df["y_true"], closed_pred)
robust_cm = make_confusion_df(df["y_true"], robust_pred)

display(closed_cm)
display(robust_cm)


,pred_B. endophyticus,pred_B_cereus,pred_M_luteus
true_B. endophyticus,0,0,0
true_B_cereus,28,0,0
true_M_luteus,25,0,0


,pred_B. endophyticus,pred_B_cereus,pred_M_luteus
true_B. endophyticus,0,0,0
true_B_cereus,28,0,0
true_M_luteus,25,0,0


## 8. Summarize predicted composition by experiment

In [14]:
predictions_df = df.copy()

predictions_df["closed_set_pred"] = closed_pred
predictions_df["robust_pred"] = robust_pred
predictions_df["confidence"] = proba.max(axis=1)
predictions_df["entropy"] = entropy_from_proba(proba)

for i, class_name in enumerate(class_names):
    predictions_df[f"prob_{class_name}"] = proba[:, i]


def composition_summary(predictions_df: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    group_cols = ["experiment_name", "y_true"] if "experiment_name" in predictions_df.columns else ["y_true"]

    counts = (
        predictions_df
        .groupby(group_cols + [pred_col])
        .size()
        .reset_index(name="n")
    )

    totals = (
        predictions_df
        .groupby(group_cols)
        .size()
        .reset_index(name="total")
    )

    out = counts.merge(totals, on=group_cols, how="left")
    out["fraction"] = out["n"] / out["total"]

    return out.sort_values(group_cols + ["fraction"], ascending=[True] * len(group_cols) + [False])


closed_composition = composition_summary(predictions_df, "closed_set_pred")
robust_composition = composition_summary(predictions_df, "robust_pred")

display(closed_composition)
display(robust_composition)


,experiment_name,y_true,closed_set_pred,n,total,fraction
0,bacillus_cereus_possible,B_cereus,B. endophyticus,28,28,1.0
1,micrococcus,M_luteus,B. endophyticus,25,25,1.0


,experiment_name,y_true,robust_pred,n,total,fraction
0,bacillus_cereus_possible,B_cereus,B. endophyticus,28,28,1.0
1,micrococcus,M_luteus,B. endophyticus,25,25,1.0


## 9. Save outputs

In [15]:
metrics_path = RESULTS_DIR / "robust_multimodal_new_data_metrics.csv"
predictions_path = RESULTS_DIR / "robust_multimodal_new_data_predictions.parquet"
closed_cm_path = RESULTS_DIR / "robust_multimodal_closed_set_confusion_matrix.csv"
robust_cm_path = RESULTS_DIR / "robust_multimodal_rejected_confusion_matrix.csv"
closed_comp_path = RESULTS_DIR / "robust_multimodal_closed_set_composition.csv"
robust_comp_path = RESULTS_DIR / "robust_multimodal_rejected_composition.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_parquet(predictions_path, index=False)
closed_cm.to_csv(closed_cm_path)
robust_cm.to_csv(robust_cm_path)
closed_composition.to_csv(closed_comp_path, index=False)
robust_composition.to_csv(robust_comp_path, index=False)

print("Saved:")
for p in [
    metrics_path,
    predictions_path,
    closed_cm_path,
    robust_cm_path,
    closed_comp_path,
    robust_comp_path,
]:
    print(" ", p)


Saved:
  C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\robust_multimodal_new_data_evaluation\robust_multimodal_new_data_metrics.csv
  C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\robust_multimodal_new_data_evaluation\robust_multimodal_new_data_predictions.parquet
  C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\robust_multimodal_new_data_evaluation\robust_multimodal_closed_set_confusion_matrix.csv
  C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\robust_multimodal_new_data_evaluation\robust_multimodal_rejected_confusion_matrix.csv
  C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\robust_multimodal_new_data_evaluation\robust_multimodal_closed_set_composition.csv
  C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\robust_multimoda

## 10. Optional ablation check

In [16]:
ABLATIONS = {
    "all_modalities": ["spectrometer", "lifetime", "scattering", "scalar"],
    "spectrometer_only": ["spectrometer"],
    "lifetime_only": ["lifetime"],
    "scattering_only": ["scattering"],
    "scalar_only": ["scalar"],
    "no_scattering": ["spectrometer", "lifetime", "scalar"],
    "no_scalar": ["spectrometer", "lifetime", "scattering"],
    "spectrometer_lifetime": ["spectrometer", "lifetime"],
}

ablation_rows = []

for ablation_id, enabled in ABLATIONS.items():
    pred, ab_proba = predict_in_batches(
        model,
        inputs,
        class_names,
        batch_size=512,
        enabled_modalities=enabled,
    )

    row = evaluate_predictions(
        df["y_true"],
        pred,
        ab_proba,
        ablation_id,
    )
    row["enabled_modalities"] = ",".join(enabled)
    ablation_rows.append(row)

ablation_df = pd.DataFrame(ablation_rows).sort_values("balanced_accuracy", ascending=False)
ablation_df.to_csv(RESULTS_DIR / "robust_multimodal_ablation_metrics.csv", index=False)

ablation_df


,model_id,n_particles,accuracy,balanced_accuracy,macro_f1,unknown_fraction,mean_confidence,median_confidence,enabled_modalities
7,spectrometer_lifetime,53,0.490566,0.477143,0.234927,0.0,0.677702,0.685533,"spectrometer,lifetime"
2,lifetime_only,53,0.471698,0.470000,0.319357,0.0,0.639191,0.630346,lifetime
1,spectrometer_only,53,0.150943,0.142857,0.097561,0.0,0.592883,0.572024,spectrometer
6,no_scalar,53,0.018868,0.017857,0.013793,0.0,0.820855,0.962023,"spectrometer,lifetime,scattering"
3,scattering_only,53,0.000000,0.000000,0.000000,0.0,0.833233,0.903823,scattering
0,all_modalities,53,0.000000,0.000000,0.000000,0.0,0.999572,1.000000,"spectrometer,lifetime,scattering,scalar"
5,no_scattering,53,0.000000,0.000000,0.000000,0.0,0.972970,0.999984,"spectrometer,lifetime,scalar"
4,scalar_only,53,0.000000,0.000000,0.000000,0.0,0.982397,0.999813,scalar
